# netMHCpan
![netMHCpan](https://proto-bio.github.io/proto-assets/images/tool/netmhcpan/hero.png)

This notebook demonstrates the two netMHCpan tools for MHC class I epitope prediction.
`run_netmhcpan_peptides` scores peptides you already have against a panel of MHC class I
alleles, and `run_netmhcpan_protein` windows a protein into class I k-mers and scores every
window, which is the usual way to screen a designed or natural sequence for epitopes.
netMHCpan is an ensemble of neural networks trained jointly on binding-affinity (BA) and
mass-spectrometry eluted-ligand (EL) data, so a single pan-specific model covers any allele
with a known sequence. It uses structural input features, offers two transfer-learned heads
for pathogen epitopes and neoepitopes (`include_pathogen`, `include_neoepitope`), and can
encode flanking context on the protein scan (`use_context`). Predictions come back as a %rank against a background of random
peptides, which is what makes them comparable across alleles.

Learn more at [services.healthtech.dtu.dk/services/NetMHCpan-4.2](https://services.healthtech.dtu.dk/services/NetMHCpan-4.2/)
and in the method paper: Nilsson, Greenbaum, Peters & Nielsen, *NetMHCpan-4.2: improved
prediction of CD8+ epitopes by use of transfer learning and structural features*,
Frontiers in Immunology 16:1616113 (2025),
[doi:10.3389/fimmu.2025.1616113](https://doi.org/10.3389/fimmu.2025.1616113).

**Note:** netMHCpan is license-gated and user-provisioned. It is not bundled with proto-tools
and cannot be downloaded automatically: the DTU academic license requires you to register and
accept the terms yourself, and forbids redistributing the package. To run the cells below you
must install netMHCpan 4.2 locally and either set the `NETMHCPAN_HOME` environment variable to
the install root or pass `netmhcpan_dir` in the config. See the toolkit's `SETUP.md` for the
provisioning steps. Every run cell in this notebook errors without a local install.

In [1]:
from proto_tools.tools import (
    NetMHCpanPeptidesConfig,
    NetMHCpanPeptidesInput,
    NetMHCpanProteinConfig,
    NetMHCpanProteinInput,
    run_netmhcpan_peptides,
    run_netmhcpan_protein,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Display input API reference
display_api_reference("netmhcpan-peptides", "input", "run_netmhcpan_peptides")

# Four well-characterised class I peptides:
#   AAAWYLWEV - HIV-1 reverse transcriptase, a known HLA-A*02:01 binder
#   GILGFVFTL - influenza A M1(58-66), the canonical HLA-A*02:01 epitope
#   KLGGALQAK - human CMV IE1(316-324), HLA-A*03:01-restricted (an A*02:01 non-binder)
#   AAGLQDCTM - no reported class I restriction; included as a negative control
inputs = NetMHCpanPeptidesInput(
    peptides=["AAAWYLWEV", "GILGFVFTL", "KLGGALQAK", "AAGLQDCTM"],
)

**Input** — `NetMHCpanPeptidesInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>peptides</code> | <code>list[str]</code> | required | MHC class I peptides (8-14 residues) to score against every allele. |

In [3]:
# Display config API reference
display_api_reference("netmhcpan-peptides", "config", "run_netmhcpan_peptides")

# A three-allele panel. netMHCpan itself accepts only the unstarred spelling, but the wrapper
# normalizes the starred forms ("HLA-A*02:01", "A*02:01") and bare human loci ("A02:01") too,
# so all the spellings in common use converge on the same allele.
config = NetMHCpanPeptidesConfig(
    alleles=["HLA-A*02:01", "HLA-A01:01", "HLA-B07:02"],
)
print(f"Normalized allele panel: {config.alleles}")

**Config** — `NetMHCpanPeptidesConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>alleles</code> | <code>list[str]</code> | <code>['HLA-A02:01']</code> | MHC class I alleles to predict against, e.g. ['HLA-A02:01', 'HLA-B07:02']. |
| <code>include_binding_affinity</code> | <code>bool</code> | <code>False</code> | Also predict affinity (-BA): adds BA score, BA %rank and nM affinity. |
| <code>include_pathogen</code> | <code>bool</code> | <code>False</code> | Also report the IEDB-finetuned pathogen head (-pathogen): score and %rank. |
| <code>include_neoepitope</code> | <code>bool</code> | <code>False</code> | Also report the CEDAR-finetuned neoepitope head (-neo): score and %rank. |
| <code>strong_rank</code> | <code>float</code> | <code>0.5</code> | EL %rank at or below which a peptide counts as a strong binder. |
| <code>weak_rank</code> | <code>float</code> | <code>2.0</code> | EL %rank at or below which a peptide counts as a weak binder. |
| <code>netmhcpan_dir</code> | <code>str &#124; None</code> | <code>None</code> | Local netMHCpan install root; overrides $NETMHCPAN_HOME and the tool cache. |

Normalized allele panel: ['HLA-A02:01', 'HLA-A01:01', 'HLA-B07:02']


In [4]:
# Display output API reference
display_api_reference("netmhcpan-peptides", "output", "run_netmhcpan_peptides")

**Output** — `NetMHCpanPeptidesOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>predictions</code> | <code>list[PeptidePrediction]</code> | required | One entry per input peptide, index-aligned with the inputs. |
| <code>netmhcpan_version</code> | <code>str &#124; None</code> | <code>None</code> | Version reported by the invoked binary; results differ across versions. |

**Metrics** — `NetMHCpanMetrics` (one set per `predictions` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>strong_binder_count</code> | <code>int</code> | <code>&gt;= 0</code> |  | Peptide-allele pairs at or below the strong %rank threshold. |
| <code>weak_binder_count</code> | <code>int</code> | <code>&gt;= 0</code> |  | Pairs between the strong and weak %rank thresholds. |
| <code>strong_binder_fraction</code> | <code>float</code> | <code>[0, 1]</code> |  | Strong binders over all pairs; comparable across panels and lengths. |
| <code>best_rank</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> | Lowest EL %rank over all pairs; the single strongest predicted binder. |
| <code>binding_propensity</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  | Mean smooth 0-1 propensity over pairs; a continuous optimization target. |
| <code>best_rank_ba</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> | Lowest binding-affinity %rank over all pairs. |
| <code>best_affinity_nm</code> | <code>float</code> | <code>&gt;= 0</code> | <code>nM</code> | Tightest predicted affinity over all pairs; lower binds more tightly. |
| <code>best_rank_pathogen</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> | Lowest pathogen-epitope %rank over all pairs. |
| <code>best_rank_neo</code> | <code>float</code> | <code>[0, 100]</code> | <code>percent</code> | Lowest neoepitope %rank over all pairs. |

In [5]:
# Score every peptide against every allele in the panel (one netMHCpan invocation).
peptide_result = run_netmhcpan_peptides(inputs, config)
print(f"netMHCpan version: {peptide_result.netmhcpan_version}")

# One PeptidePrediction per input peptide, index-aligned with inputs.peptides. Each carries
# one binding record per configured allele, plus that peptide's summary metrics.
for prediction in peptide_result.predictions:
    print(f"\n{prediction.peptide}")
    for binding in prediction.bindings:
        print(
            f"  {binding.allele:<12} rank_el={binding.rank_el:>7.3f}  "
            f"score_el={binding.score_el:.6f}  {binding.binder_class}"
        )
    print(
        f"  summary: strong={prediction.strong_binder_count} weak={prediction.weak_binder_count} "
        f"best_rank={prediction.best_rank:.3f} binding_propensity={prediction.binding_propensity:.4f}"
    )

Running run_netmhcpan_peptides [00:00]

netMHCpan version: 4.2e

AAAWYLWEV
  HLA-A02:01   rank_el=  0.680  score_el=0.364671  weak
  HLA-A01:01   rank_el= 17.162  score_el=0.001290  non-binder
  HLA-B07:02   rank_el= 16.156  score_el=0.000667  non-binder
  summary: strong=0 weak=1 best_rank=0.680 binding_propensity=0.3083

GILGFVFTL
  HLA-A02:01   rank_el=  0.197  score_el=0.711032  strong
  HLA-A01:01   rank_el= 35.989  score_el=0.000308  non-binder
  HLA-B07:02   rank_el= 17.332  score_el=0.000566  non-binder
  summary: strong=1 weak=0 best_rank=0.197 binding_propensity=0.3355

KLGGALQAK
  HLA-A02:01   rank_el=  9.086  score_el=0.005325  non-binder
  HLA-A01:01   rank_el= 15.781  score_el=0.001494  non-binder
  HLA-B07:02   rank_el= 29.822  score_el=0.000144  non-binder
  summary: strong=0 weak=0 best_rank=9.086 binding_propensity=0.0222

AAGLQDCTM
  HLA-A02:01   rank_el= 45.317  score_el=0.000063  non-binder
  HLA-A01:01   rank_el= 30.987  score_el=0.000423  non-binder
  HLA-B07:02   rank_el= 11.826  score_el=0.001358  no

In [6]:
# Human beta-2 microglobulin (mature chain, 99 residues) - the class I light chain itself.
B2M = (
    "IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDW"
    "SFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"
)

# The protein tool does the windowing for you: every k-mer of every configured length is
# enumerated and scored. Here 9- and 10-mers against two alleles.
protein_inputs = NetMHCpanProteinInput(sequences=[B2M])
protein_config = NetMHCpanProteinConfig(
    alleles=["HLA-A02:01", "HLA-B07:02"],
    peptide_lengths=[9, 10],
)
protein_result = run_netmhcpan_protein(protein_inputs, protein_config)

scan = protein_result.results[0]
print(f"Sequence length: {scan.sequence_length} residues")
print(f"Windows scored:  {scan.num_windows}")
print(f"Binding records: {len(scan.bindings)}  (one per window per allele)")
print(
    f"Strong binders: {scan.strong_binder_count}, weak: {scan.weak_binder_count}, "
    f"best rank: {scan.best_rank:.3f}"
)
print(f"Strong binders per 100 residues per allele: {scan.strong_binders_per_100_residues:.2f}")

# The strongest predicted epitopes. `position` is 1-indexed against the supplied sequence.
print("\nTop 8 windows by EL %rank:")
for binding in sorted(scan.bindings, key=lambda b: b.rank_el)[:8]:
    print(
        f"  pos {binding.position:>3}  {binding.peptide:<11} {binding.allele:<12} "
        f"rank_el={binding.rank_el:>6.3f}  core={binding.core:<10} {binding.binder_class}"
    )

Running run_netmhcpan_protein [00:00]

Sequence length: 99 residues
Windows scored:  181
Binding records: 362  (one per window per allele)
Strong binders: 2, weak: 2, best rank: 0.195
Strong binders per 100 residues per allele: 1.01

Top 8 windows by EL %rank:
  pos  13  HPAENGKSNF  HLA-B07:02   rank_el= 0.195  core=HPAENGKSF  strong
  pos  31  HPSDIEVDL   HLA-B07:02   rank_el= 0.276  core=HPSDIEVDL  strong
  pos  31  HPSDIEVDLL  HLA-B07:02   rank_el= 0.833  core=HPSDIEVLL  weak
  pos  63  YLLYYTEFT   HLA-A02:01   rank_el= 1.742  core=YLLYYTEFT  weak
  pos  79  ACRVNHVTL   HLA-B07:02   rank_el= 2.101  core=ACRVNHVTL  non-binder
  pos  56  FSKDWSFYL   HLA-A02:01   rank_el= 2.200  core=FSKDWSFYL  non-binder
  pos  26  YVSGFHPSDI  HLA-A02:01   rank_el= 2.651  core=YVSGFHPSI  non-binder
  pos   1  IQRTPKIQV   HLA-B07:02   rank_el= 3.152  core=IQRTPKIQV  non-binder


In [7]:
# Advanced usage: a wider allele panel, binding affinity alongside the default eluted-ligand
# output, the two 4.2 transfer-learned heads, and stricter binder thresholds.
#
#   include_binding_affinity=True passes netMHCpan's -BA flag, which *adds* score_ba, rank_ba
#   and affinity_nm to the EL columns rather than replacing them.
#   include_pathogen / include_neoepitope are additive in the same way, each adding its own
#   score and %rank from a head fine-tuned on IEDB pathogen epitopes or CEDAR neoepitopes.
#   The authors report the fine-tuning gain is modest and does not transfer between the two,
#   so match the head to the task and keep ranking on rank_el.
#   strong_rank / weak_rank only change how rank_el is classified; they do not change any score.
advanced_config = NetMHCpanPeptidesConfig(
    alleles=["HLA-A*02:01", "HLA-A01:01", "HLA-A03:01", "HLA-B07:02"],
    include_binding_affinity=True,
    include_pathogen=True,
    include_neoepitope=True,
    strong_rank=0.2,
    weak_rank=1.0,
)
advanced_result = run_netmhcpan_peptides(inputs, advanced_config)

for prediction in advanced_result.predictions:
    print(f"\n{prediction.peptide}")
    for binding in prediction.bindings:
        print(
            f"  {binding.allele:<12} rank_el={binding.rank_el:>7.3f}  "
            f"rank_ba={binding.rank_ba:>7.3f}  affinity={binding.affinity_nm:>9.2f} nM  "
            f"rank_pathogen={binding.rank_pathogen:>7.3f}  rank_neo={binding.rank_neo:>7.3f}  "
            f"{binding.binder_class}"
        )

# The BA fields are populated only when -BA was requested; in the default EL-only run above
# they are None. Same peptide, same allele, two runs:
default_binding = peptide_result.predictions[0].bindings[0]
ba_binding = advanced_result.predictions[0].bindings[0]
print(f"peptide={default_binding.peptide} allele={default_binding.allele}")
print(f"  EL-only run: rank_el={default_binding.rank_el}, rank_ba={default_binding.rank_ba}, "
      f"affinity_nm={default_binding.affinity_nm}")
print(f"  -BA run:     rank_el={ba_binding.rank_el}, rank_ba={ba_binding.rank_ba}, "
      f"affinity_nm={ba_binding.affinity_nm}")

# rank_el is identical across the two runs: -BA adds columns, it does not change the EL
# prediction. What did change is binder_class, because the thresholds were tightened.
print(f"\nrank_el unchanged: {default_binding.rank_el == ba_binding.rank_el}")
print(
    f"binder_class at strong_rank=0.5: {default_binding.binder_class}  ->  "
    f"at strong_rank=0.2: {ba_binding.binder_class}"
)

Running run_netmhcpan_peptides [00:00]


AAAWYLWEV
  HLA-A02:01   rank_el=  0.680  rank_ba=  0.060  affinity=     8.76 nM  rank_pathogen=  0.009  rank_neo=  0.179  weak
  HLA-A01:01   rank_el= 17.162  rank_ba=  3.384  affinity= 13516.19 nM  rank_pathogen=  0.103  rank_neo= 11.181  non-binder
  HLA-A03:01   rank_el= 16.243  rank_ba= 13.312  affinity= 19062.49 nM  rank_pathogen=  0.225  rank_neo= 13.391  non-binder
  HLA-B07:02   rank_el= 16.156  rank_ba=  9.710  affinity= 17103.40 nM  rank_pathogen=  0.055  rank_neo=  9.658  non-binder

GILGFVFTL
  HLA-A02:01   rank_el=  0.197  rank_ba=  0.084  affinity=    10.68 nM  rank_pathogen=  0.018  rank_neo=  0.042  strong
  HLA-A01:01   rank_el= 35.989  rank_ba= 33.120  affinity= 33931.50 nM  rank_pathogen=  4.930  rank_neo= 38.767  non-binder
  HLA-A03:01   rank_el=  7.477  rank_ba=  8.762  affinity= 13616.04 nM  rank_pathogen=  0.124  rank_neo=  6.351  non-binder
  HLA-B07:02   rank_el= 17.332  rank_ba= 20.660  affinity= 26624.49 nM  rank_pathogen=  0.074  rank_neo=  5.752  non-bin

In [8]:
# 4.2 can encode the residues flanking each window (netMHCpan's -context), which carry a
# processing signal the peptide alone does not. It is available on the protein scan because
# the flanks are read straight off the sequence you supplied; peptide mode would require you
# to provide each peptide's flanks yourself.
#
# Context re-scores every window, so a context run and a plain run are not comparable -
# pick one per experiment rather than mixing their numbers.
context_config = NetMHCpanProteinConfig(
    alleles=["HLA-A02:01", "HLA-B07:02"],
    peptide_lengths=[9, 10],
    use_context=True,
)
context_scan = run_netmhcpan_protein(protein_inputs, context_config).results[0]

print(f"Windows scored: {context_scan.num_windows} (unchanged by context)")
print(
    f"Strong binders without context: {scan.strong_binder_count}, "
    f"with context: {context_scan.strong_binder_count}"
)
print(f"Best rank without context: {scan.best_rank:.3f}, with context: {context_scan.best_rank:.3f}")

# The same windows, re-ranked. Position and peptide are unchanged; the scores are not.
print("\nTop 5 windows with context encoding:")
for binding in sorted(context_scan.bindings, key=lambda b: b.rank_el)[:5]:
    print(
        f"  pos {binding.position:>3}  {binding.peptide:<11} {binding.allele:<12} "
        f"rank_el={binding.rank_el:>6.3f}  {binding.binder_class}"
    )


Running run_netmhcpan_protein [00:00]

Windows scored: 181 (unchanged by context)
Strong binders without context: 2, with context: 2
Best rank without context: 0.195, with context: 0.177

Top 5 windows with context encoding:
  pos  13  HPAENGKSNF  HLA-B07:02   rank_el= 0.177  strong
  pos  31  HPSDIEVDL   HLA-B07:02   rank_el= 0.232  strong
  pos  31  HPSDIEVDLL  HLA-B07:02   rank_el= 0.545  weak
  pos  91  KIVKWDRDM   HLA-B07:02   rank_el= 0.759  weak
  pos   1  IQRTPKIQV   HLA-B07:02   rank_el= 1.051  weak


In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# CSV gives a flat table, one row per peptide-allele (or window-allele) prediction - the form
# you want for downstream filtering and plotting.
advanced_result.export(name="netmhcpan_peptides", export_path=output_dir, file_format="csv")
protein_result.export(name="netmhcpan_protein_scan", export_path=output_dir, file_format="csv")

# JSON preserves the full nested structure: per-item summary metrics, every binding record,
# and the netMHCpan version that produced them.
advanced_result.export(name="netmhcpan_peptides", export_path=output_dir, file_format="json")

for path in sorted(output_dir.glob("netmhcpan_*")):
    print(f"{path.name}  ({path.stat().st_size} bytes)")

print("\nFirst rows of the peptide CSV:")
print("\n".join((output_dir / "netmhcpan_peptides.csv").read_text().splitlines()[:5]))

netmhcpan_peptides.csv  (1743 bytes)
netmhcpan_peptides.json  (10357 bytes)
netmhcpan_protein_scan.csv  (21802 bytes)

First rows of the peptide CSV:
peptide,allele,score_el,rank_el,binder_class,score_ba,rank_ba,affinity_nm,score_pathogen,rank_pathogen,score_neo,rank_neo
AAAWYLWEV,HLA-A02:01,0.364671,0.68,weak,0.799447,0.06,8.76,0.800747,0.009,0.351067,0.179
AAAWYLWEV,HLA-A01:01,0.00129,17.162,non-binder,0.120902,3.384,13516.19,0.557724,0.103,0.002177,11.181
AAAWYLWEV,HLA-A03:01,0.000769,16.243,non-binder,0.089124,13.312,19062.49,0.344755,0.225,0.001826,13.391
AAAWYLWEV,HLA-B07:02,0.000667,16.156,non-binder,0.099147,9.71,17103.4,0.639716,0.055,0.002539,9.658
